# ForJustice3K: Google Colab Training Walkthrough

This notebook fine-tunes the WangchanBERTa-based hatespeech detector from the ForJustice3K project, then runs an inference demo against a sample sentence. Execute the cells top-to-bottom in a fresh Colab runtime.

⚠️ For faster training, switch *Runtime → Change runtime type → GPU*. The script autodetects the device.

In [ ]:
# Optional: inspect the accelerator that Colab attached to this runtime.
!nvidia-smi

In [ ]:
import pathlib
import shutil

repo_url = "https://github.com/psrisuphan/4j3k.git"
repo_dir = pathlib.Path("4j3k")

if repo_dir.exists():
    print(f"Removing existing clone at {repo_dir.resolve()} ...")
    shutil.rmtree(repo_dir)

!git clone {repo_url}
%cd 4j3k

In [ ]:
# Install project dependencies. Expect the first run to take a few minutes.
!pip install -q -U pip
!pip install -q -r requirements.txt

In [ ]:
# Fine-tune the classifier with Colab-friendly defaults.
!python train_model.py \
    --data data/HateThaiSent.csv \
    --extra-data data/ThaiToxicityTweet_converted.csv \
    --output-dir models/wangchanberta-colab \
    --epochs 5 \
    --batch-size 4 \
    --gradient-accumulation 2 \
    --learning-rate 2e-5 \
    --lr-scheduler cosine \
    --warmup-ratio 0.1 \
    --weight-decay 0.01 \
    --gradient-checkpointing \
    --group-by-length \
    --eval-size 0.1 \
    --test-size 0.1 \
    --device auto \
    --max-gpu-memory-fraction 0.85

After training finishes, review the evaluation metrics captured in `models/wangchanberta-colab/eval_metrics.json`.

In [ ]:
!cat models/wangchanberta-colab/eval_metrics.json

In [ ]:
import subprocess

sample_text = "ที่นี่เป็นข้อความตัวอย่างสำหรับการทดสอบระบบตรวจจับคำพูดแสดงความเกลียดชัง"
age = 15

print("Scoring sample text ...")
result = subprocess.check_output([
    "python",
    "predict.py",
    sample_text,
    "--age",
    str(age),
    "--model",
    "models/wangchanberta-colab"
])

print(result.decode("utf-8"))

## Batch scoring with `sample_sentences.jsonl`

Use the helper script to evaluate the bundled demo prompts or your own JSONL/text file. The command below also captures a CSV under `predictions.csv` for later inspection.

In [ ]:
!python sample_predictions.py \
    --model models/wangchanberta-colab \
    --input sample_sentences.jsonl \
    --age 15 \
    --output predictions.csv